# Risk Assessment Engine

## Purpose

The Risk Assessment Engine evaluates environmental conditions and converts them into standardized risk scores.

Version 1 assesses:

- Flood Risk
- Heat Risk
- Vegetation Stress
- Urban Exposure

The resulting Risk Assessment Product supports Earth Intelligence analyses and decision-making.

# Import Libraries

## Purpose

Import the libraries required for risk assessment and analysis.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

# Load Environmental Indicator Product

## Purpose

Load the Environmental Indicator Product generated by the Environmental Indicator Engine.

The product provides standardized environmental indicators used to calculate thematic environmental risks.

In [2]:
DATA_DIR = Path(
    "/Users/ShreyaJariwalaMain/_GeoAI_Notebook/Earth-Intelligence-System/data/outputs"
)

environmental_indicator_product_file = (

    DATA_DIR

    / "environmental_indicator_product.json"

)

with open(

    environmental_indicator_product_file,

    "r",

    encoding="utf-8"

) as file:

    environmental_indicator_product = json.load(file)

print("Environmental Indicator Product loaded successfully.")

Environmental Indicator Product loaded successfully.


# Risk Framework

## Purpose

Define the environmental variables used to assess each risk category.

The Risk Framework provides a transparent mapping between environmental indicators and the corresponding risk assessments. This framework serves as the foundation for all risk calculations performed by the Risk Assessment Engine.

In [3]:
risk_framework = {

    "Flood Risk": {

        "inputs": [

            "Mean Elevation",

            "Mean Slope",

            "Precipitation",

            "Water Presence"

        ]

    },

    "Heat Risk": {

        "inputs": [

            "Mean Temperature",

            "Built-up Area",

            "Vegetation"

        ]

    },

    "Vegetation Stress": {

        "inputs": [

            "NDVI",

            "Precipitation"

        ]

    },

    "Urban Exposure": {

        "inputs": [

            "Built-up Area"

        ]

    }

}

risk_framework

{'Flood Risk': {'inputs': ['Mean Elevation',
   'Mean Slope',
   'Precipitation',
   'Water Presence']},
 'Heat Risk': {'inputs': ['Mean Temperature', 'Built-up Area', 'Vegetation']},
 'Vegetation Stress': {'inputs': ['NDVI', 'Precipitation']},
 'Urban Exposure': {'inputs': ['Built-up Area']}}

# Flood Risk Assessment

## Purpose

Assess the potential flood risk within the Area of Interest.

Flood Risk is estimated using environmental indicators that influence surface water accumulation and runoff, including:

- Elevation
- Slope
- Precipitation
- Water Presence

The resulting score is classified into a standardized risk category.

In [4]:
mean_elevation = environmental_indicator_product["summary"][
    "mean_elevation"
]

mean_slope = environmental_indicator_product["terrain"][
    "Slope"
]["Mean"]

precipitation = environmental_indicator_product["weather"][
    "Precipitation (mm)"
]["Total"]

water_percentage = 0

for land_cover in environmental_indicator_product[
    "land_cover"
]["Summary"]:

    if land_cover["class"] == "Permanent Water":

        water_percentage = land_cover["percentage"]

        break

# Flood Risk Calculation

## Purpose

Calculate a standardized Flood Risk score.

Each environmental factor contributes to the overall Flood Risk score based on simple rule-based criteria.

Version 1 uses an interpretable scoring approach designed for demonstration and future refinement.

In [5]:
flood_score = 0

# Elevation
if mean_elevation < 20:
    flood_score += 30

elif mean_elevation < 50:
    flood_score += 15

# Slope
if mean_slope < 5:
    flood_score += 25

elif mean_slope < 15:
    flood_score += 10

# Rainfall
if precipitation > 150:
    flood_score += 30

elif precipitation > 75:
    flood_score += 15

# Water Presence
if water_percentage > 10:
    flood_score += 15

elif water_percentage > 5:
    flood_score += 8

flood_score = min(
    flood_score,
    100
)

if flood_score <= 25:

    flood_category = "Low"

elif flood_score <= 50:

    flood_category = "Moderate"

elif flood_score <= 75:

    flood_category = "High"

else:

    flood_category = "Very High"

flood_risk = {

    "Score": flood_score,

    "Category": flood_category

}

flood_risk

{'Score': 75, 'Category': 'High'}

# Heat Risk Assessment

## Purpose

Assess the potential heat risk within the Area of Interest.

Heat Risk is estimated using environmental indicators that influence urban heat accumulation, including:

- Temperature
- Vegetation
- Built-up Area

The resulting score is classified into a standardized risk category.

In [6]:
mean_temperature = environmental_indicator_product["summary"][
    "mean_temperature"
]

mean_ndvi = environmental_indicator_product["summary"][
    "mean_ndvi"
]

built_up_percentage = 0

for land_cover in environmental_indicator_product[
    "land_cover"
]["Summary"]:

    if land_cover["class"] == "Built-up":

        built_up_percentage = land_cover["percentage"]

        break

# Heat Risk Calculation

## Purpose

Calculate a standardized Heat Risk score.

The Heat Risk score combines atmospheric conditions, vegetation cover, and urbanization to estimate the potential for elevated surface temperatures.

Version 1 uses a transparent rule-based scoring approach.

In [7]:
heat_score = 0

heat_drivers = []

# Temperature
if mean_temperature > 35:

    heat_score += 40

    heat_drivers.append(
        "High temperature"
    )

elif mean_temperature > 30:

    heat_score += 20

    heat_drivers.append(
        "Warm temperature"
    )

# Vegetation
if mean_ndvi < 0.20:

    heat_score += 30

    heat_drivers.append(
        "Low vegetation"
    )

elif mean_ndvi < 0.40:

    heat_score += 15

    heat_drivers.append(
        "Moderate vegetation"
    )

# Built-up Area
if built_up_percentage > 60:

    heat_score += 30

    heat_drivers.append(
        "Highly urbanized"
    )

elif built_up_percentage > 40:

    heat_score += 15

    heat_drivers.append(
        "Moderately urbanized"
    )

heat_score = min(
    heat_score,
    100
)

if heat_score <= 25:

    heat_category = "Low"

elif heat_score <= 50:

    heat_category = "Moderate"

elif heat_score <= 75:

    heat_category = "High"

else:

    heat_category = "Very High"

heat_risk = {

    "Score": heat_score,

    "Category": heat_category,

    "Drivers": heat_drivers

}

heat_risk

{'Score': 30, 'Category': 'Moderate', 'Drivers': ['Low vegetation']}

# Vegetation Stress Assessment

## Purpose

Assess the potential vegetation stress within the Area of Interest.

Vegetation Stress is estimated using vegetation condition and precipitation.

Healthy vegetation supported by sufficient rainfall indicates lower stress, while sparse vegetation and limited precipitation indicate higher stress.

The resulting score is classified into a standardized risk category.

In [8]:
mean_ndvi = environmental_indicator_product["summary"][
    "mean_ndvi"
]

precipitation = environmental_indicator_product["weather"][
    "Precipitation (mm)"
]["Total"]

# Vegetation Stress Calculation

## Purpose

Calculate a standardized Vegetation Stress score.

The score combines vegetation condition and precipitation using a transparent rule-based approach.

Version 1 provides an interpretable estimate of vegetation stress for the selected Area of Interest.

In [9]:
vegetation_stress_score = 0

vegetation_stress_drivers = []

# Vegetation Condition
if mean_ndvi < 0.20:

    vegetation_stress_score += 60

    vegetation_stress_drivers.append(
        "Low vegetation condition"
    )

elif mean_ndvi < 0.40:

    vegetation_stress_score += 30

    vegetation_stress_drivers.append(
        "Moderate vegetation condition"
    )

# Rainfall
if precipitation < 25:

    vegetation_stress_score += 40

    vegetation_stress_drivers.append(
        "Low precipitation"
    )

elif precipitation < 75:

    vegetation_stress_score += 20

    vegetation_stress_drivers.append(
        "Moderate precipitation"
    )

vegetation_stress_score = min(

    vegetation_stress_score,

    100

)

if vegetation_stress_score <= 25:

    vegetation_stress_category = "Low"

elif vegetation_stress_score <= 50:

    vegetation_stress_category = "Moderate"

elif vegetation_stress_score <= 75:

    vegetation_stress_category = "High"

else:

    vegetation_stress_category = "Very High"

vegetation_stress = {

    "Score": vegetation_stress_score,

    "Category": vegetation_stress_category,

    "Drivers": vegetation_stress_drivers

}

vegetation_stress

{'Score': 60, 'Category': 'High', 'Drivers': ['Low vegetation condition']}

# Urban Exposure Assessment

## Purpose

Assess the degree of urban exposure within the Area of Interest.

Urban Exposure is estimated using the proportion of built-up land cover.

Higher levels of urbanization generally indicate greater exposure of infrastructure, population, and assets to environmental hazards.

In [10]:
built_up_percentage = 0

for land_cover in environmental_indicator_product[
    "land_cover"
]["Summary"]:

    if land_cover["class"] == "Built-up":

        built_up_percentage = land_cover["percentage"]

        break

# Urban Exposure Calculation

## Purpose

Calculate a standardized Urban Exposure score.

The score is derived from the proportion of built-up land within the Area of Interest.

Version 1 provides an interpretable estimate of urban exposure based on land cover composition.

In [11]:
urban_exposure_score = 0

urban_exposure_drivers = []

if built_up_percentage > 60:

    urban_exposure_score = 100

    urban_exposure_drivers.append(
        "Highly urbanized"
    )

elif built_up_percentage > 40:

    urban_exposure_score = 75

    urban_exposure_drivers.append(
        "Moderately urbanized"
    )

elif built_up_percentage > 20:

    urban_exposure_score = 50

    urban_exposure_drivers.append(
        "Partially urbanized"
    )

else:

    urban_exposure_score = 25

    urban_exposure_drivers.append(
        "Limited urban development"
    )

if urban_exposure_score <= 25:

    urban_exposure_category = "Low"

elif urban_exposure_score <= 50:

    urban_exposure_category = "Moderate"

elif urban_exposure_score <= 75:

    urban_exposure_category = "High"

else:

    urban_exposure_category = "Very High"

urban_exposure = {

    "Score": urban_exposure_score,

    "Category": urban_exposure_category,

    "Drivers": urban_exposure_drivers

}

urban_exposure

{'Score': 50, 'Category': 'Moderate', 'Drivers': ['Partially urbanized']}

# Risk Assessment Product

## Purpose

Create the standardized Risk Assessment Product.

The Risk Assessment Product combines all thematic environmental risks into a single structured product.

This product serves as the primary input for the Earth Intelligence Index Engine.

In [12]:
from datetime import datetime

risk_assessment_product = {

    "metadata": {

        "engine": "Risk Assessment Engine",

        "version": "1.0",

        "created_at": datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    },

    "summary": {

        "highest_risk": max(

            [

                flood_risk,

                heat_risk,

                vegetation_stress,

                urban_exposure

            ],

            key=lambda risk: risk["Score"]

        )["Category"]

    },

    "flood_risk": flood_risk,

    "heat_risk": heat_risk,

    "vegetation_stress": vegetation_stress,

    "urban_exposure": urban_exposure

}

risk_assessment_product

{'metadata': {'engine': 'Risk Assessment Engine',
  'version': '1.0',
  'created_at': '2026-07-16 01:45:54'},
 'summary': {'highest_risk': 'High'},
 'flood_risk': {'Score': 75, 'Category': 'High'},
 'heat_risk': {'Score': 30,
  'Category': 'Moderate',
  'Drivers': ['Low vegetation']},
 'vegetation_stress': {'Score': 60,
  'Category': 'High',
  'Drivers': ['Low vegetation condition']},
 'urban_exposure': {'Score': 50,
  'Category': 'Moderate',
  'Drivers': ['Partially urbanized']}}

# Export Risk Assessment Product

## Purpose

Export the Risk Assessment Product for downstream Earth Intelligence analysis.

The product is saved as a JSON file and will be used by the Earth Intelligence Index Engine.

In [13]:
from pathlib import Path
import json

OUTPUT_DIR = Path(
    "/Users/ShreyaJariwalaMain/_GeoAI_Notebook/Earth-Intelligence-System/data/outputs"
)

risk_assessment_product_file = (

    OUTPUT_DIR

    / "risk_assessment_product.json"

)

with open(

    risk_assessment_product_file,

    "w",

    encoding="utf-8"

) as file:

    json.dump(

        risk_assessment_product,

        file,

        indent=4,

        default=str

    )

print(

    f"Risk Assessment Product : "

    f"{risk_assessment_product_file.name}"

)

Risk Assessment Product : risk_assessment_product.json


# Risk Assessment Engine Summary

## Purpose

Summarize the environmental risks identified for the selected Area of Interest.

This summary confirms that the Risk Assessment Product has been successfully generated and is ready for Earth Intelligence analysis.

In [14]:
summary = {

    "Flood Risk": (

        f"{flood_risk['Category']} "

        f"({flood_risk['Score']})"

    ),

    "Heat Risk": (

        f"{heat_risk['Category']} "

        f"({heat_risk['Score']})"

    ),

    "Vegetation Stress": (

        f"{vegetation_stress['Category']} "

        f"({vegetation_stress['Score']})"

    ),

    "Urban Exposure": (

        f"{urban_exposure['Category']} "

        f"({urban_exposure['Score']})"

    )

}

for key, value in summary.items():

    print(f"{key:<22}: {value}")

Flood Risk            : High (75)
Heat Risk             : Moderate (30)
Vegetation Stress     : High (60)
Urban Exposure        : Moderate (50)
